# ASCAD: approved 45k baseline

Account: nmavros. One additional baseline only, total cap5.
Same CNN16952 / seed0 / 50epochs / batch50 / author LR cycle.
Training45k / shared validation5k / training-only MinMax. No combined or attack.

## Before execution
1. Upload kaggle_resume_input_literature.zip as a PRIVATE Dataset and attach it with Add Input.
2. Enable Internet and GPU (prefer Tesla T4). Keep notebook private.
3. Execute once using Save Version > Save & Run All. Do not run interactively first.
4. Download budget_diagnostic_results.zip from Output after completion.

The input archive contains code, official ASCAD and the preserved original checkpoint. No API key is needed inside this notebook. The original con1los study is preserved.


In [ ]:
"""Discover project/data and restore existing runs before importing the training package."""
import hashlib
import json
import shutil
import stat
import zipfile
from pathlib import Path, PurePosixPath


def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def project_sha256(root):
    root = Path(root)
    files = sorted([*root.glob("src/**/*.py"), *root.glob("scripts/*.py"),
                    *root.glob("configs/*.json"), root / "pyproject.toml"])
    digest = hashlib.sha256()
    for path in files:
        digest.update(path.relative_to(root).as_posix().encode())
        digest.update(path.read_bytes())
    return digest.hexdigest()


def archive_project_sha256(path):
    with zipfile.ZipFile(path) as archive:
        names = archive.namelist()
        required = {"pyproject.toml", "src/sca/train.py", "requirements.txt", "tests/test_core.py"}
        if not required.issubset(names):
            return None
        selected = sorted(name for name in names if name == "pyproject.toml" or
                          (name.startswith("src/") and name.endswith(".py")) or
                          (name.startswith("scripts/") and name.count("/") == 1 and name.endswith(".py")) or
                          (name.startswith("configs/") and name.count("/") == 1 and name.endswith(".json")))
        digest = hashlib.sha256()
        for name in selected:
            digest.update(name.encode())
            digest.update(archive.read(name))
        return digest.hexdigest()


def extract_checked(path, destination):
    destination = Path(destination).resolve()
    with zipfile.ZipFile(path) as archive:
        if sum(member.file_size for member in archive.infolist()) > 256 * 1024 * 1024:
            raise ValueError("Unexpectedly large project/run input archive")
        for member in archive.infolist():
            relative = PurePosixPath(member.filename)
            if (relative.is_absolute() or ".." in relative.parts or
                    "\\" in member.filename or ":" in member.filename or
                    stat.S_ISLNK(member.external_attr >> 16) or
                    not (destination / member.filename).resolve().is_relative_to(destination)):
                raise ValueError("Unsafe archive member: " + member.filename)
        destination.mkdir(parents=True, exist_ok=True)
        archive.extractall(destination)


def bootstrap_inputs(input_root, project_root, run_root, expected_dataset_sha256,
                     expected_source_sha256=None, require_checkpoint=True):
    input_root, project_root, run_root = map(Path, (input_root, project_root, run_root))
    staging = project_root.parent / "input_assets"
    search_roots = [input_root]
    print("Input folders:", [str(p) for p in sorted(input_root.iterdir())] if input_root.exists() else [])
    containers = [p for p in input_root.rglob("*.zip")
                  if p.stem.startswith(("kaggle_resume_input", "results"))]
    for container in sorted(containers):
        extracted = staging / file_sha256(container)[:16]
        marker = extracted / ".bootstrap_complete"
        if not marker.exists():
            extract_checked(container, extracted)
            marker.write_text(file_sha256(container), encoding="utf-8")
        search_roots.append(extracted)
    code_candidates, dataset_candidates, restore_candidates, restore_folders = [], [], [], []
    for search_root in search_roots:
        for path in search_root.rglob("pyproject.toml"):
            candidate = path.parent
            if (candidate / "src/sca/train.py").is_file() and (candidate / "requirements.txt").is_file():
                code_candidates.append((candidate, project_sha256(candidate), False))
        for path in search_root.rglob("*.zip"):
            identity = archive_project_sha256(path)
            if identity is not None:
                code_candidates.append((path, identity, True))
        dataset_candidates.extend(search_root.rglob("ASCAD*.h5"))
        restore_candidates.extend(search_root.rglob("sca_runs_*.zip"))
        restore_folders.extend(path.parent.parent for path in search_root.rglob("last.pt")
                               if path.parent.name == "none_seed0" and
                               (path.parent / "manifest.json").is_file() and
                               (path.parent / "history.json").is_file())
    if (project_root / "src/sca/train.py").is_file():
        code_candidates.append((project_root, project_sha256(project_root), False))
    print("Code sources:", [(str(p), h) for p, h, _ in code_candidates])
    print("ASCAD files:", [str(p) for p in dataset_candidates])
    print("Restore archives:", [str(p) for p in restore_candidates])
    print("Extracted run folders:", [str(p) for p in restore_folders])
    valid_datasets = [p for p in sorted(set(dataset_candidates))
                      if file_sha256(p) == expected_dataset_sha256]
    if not valid_datasets:
        raise RuntimeError("ASCAD.h5 is missing or has the wrong checksum. Add the original data Input or kaggle_resume_input.zip.")
    restores = {file_sha256(p): p for p in sorted(set(restore_candidates))}
    if len(restores) > 1 and not (run_root / "none_seed0/last.pt").exists():
        raise RuntimeError("Different restore archives are attached. Keep only the intended checkpoint Input.")
    restore = next(iter(restores.values()), None)
    folders = {file_sha256(p / "none_seed0/last.pt"): p for p in sorted(set(restore_folders))}
    checkpoint_hashes = set(folders)
    for archive_path in restores.values():
        with zipfile.ZipFile(archive_path) as archive:
            checkpoint_hashes.add(hashlib.sha256(archive.read("none_seed0/last.pt")).hexdigest())
    if len(checkpoint_hashes) > 1 and not (run_root / "none_seed0/last.pt").exists():
        raise RuntimeError("Different restore checkpoints are attached. Keep only the intended checkpoint Input.")
    restore_folder = next(iter(folders.values()), None)
    manifest_path = run_root / "none_seed0/manifest.json"
    if manifest_path.exists():
        checkpoint_source = json.loads(manifest_path.read_text())["code"]["source_sha256"]
    elif restore is not None:
        with zipfile.ZipFile(restore) as archive:
            checkpoint_source = json.loads(archive.read("none_seed0/manifest.json"))["code"]["source_sha256"]
    elif restore_folder is not None:
        checkpoint_source = json.loads((restore_folder / "none_seed0/manifest.json").read_text())["code"]["source_sha256"]
    else:
        checkpoint_source = expected_source_sha256
    if expected_source_sha256 and checkpoint_source != expected_source_sha256:
        raise RuntimeError("The checkpoint uses a different code version.")
    matches = [(p, h, is_archive) for p, h, is_archive in code_candidates
               if checkpoint_source is None or h == checkpoint_source]
    if not matches:
        raise RuntimeError("Project code is missing or does not match the checkpoint. Add kaggle_project.zip or kaggle_resume_input.zip.")
    if len({h for _, h, _ in matches}) > 1:
        raise RuntimeError("Different code versions are attached. Select the intended project Input.")
    candidate, identity, is_archive = matches[0]
    if project_root.exists():
        if not (project_root / "src/sca/train.py").is_file() or project_sha256(project_root) != identity:
            raise RuntimeError("Existing working project is incomplete/different. Use a fresh session with the resume Input.")
    elif is_archive:
        extract_checked(candidate, project_root)
    else:
        shutil.copytree(candidate, project_root)
    if project_sha256(project_root) != identity:
        raise RuntimeError("Extracted project fingerprint mismatch")
    last_checkpoint = run_root / "none_seed0/last.pt"
    if not last_checkpoint.exists() and (restore is not None or restore_folder is not None):
        if run_root.exists() and any(run_root.iterdir()):
            raise RuntimeError("Existing run folder is incomplete; refusing to overwrite it.")
        if restore is not None:
            extract_checked(restore, run_root)
        else:
            run_root.mkdir(parents=True, exist_ok=True)
            for name in ("none_seed0", "noise_seed0", "shift_seed0", "combined_seed0"):
                if (restore_folder / name).is_dir():
                    shutil.copytree(restore_folder / name, run_root / name)
            for name in ("study_summary.json", "protocol_freeze.json"):
                if (restore_folder / name).is_file():
                    shutil.copy2(restore_folder / name, run_root / name)
    if require_checkpoint and not last_checkpoint.exists():
        raise RuntimeError("Checkpoint Input is missing. Add sca_runs_minimal_v1.zip or kaggle_resume_input.zip to continue baseline.")
    if last_checkpoint.exists():
        restored = json.loads((run_root / "none_seed0/manifest.json").read_text())
        if restored["code"]["source_sha256"] != identity or restored["dataset_sha256"] != expected_dataset_sha256:
            raise RuntimeError("Restored run does not match the selected code/data")
        history = json.loads((run_root / "none_seed0/history.json").read_text())
        print("Existing checkpoint kept. Completed epoch:", history[-1]["epoch"])
    print("Project:", project_root, "Dataset:", valid_datasets[0], "Source SHA256:", identity)
    return valid_datasets[0]

import os
import sys
import subprocess
import time

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
PROJECT_ROOT = Path("/kaggle/working/hardware_sca")
INPUT_ROOT = Path("/kaggle/input")
RUN_ROOT = Path("/kaggle/working/runs/minimal_v3_literature")
DATASET_PATH = bootstrap_inputs(
    INPUT_ROOT, PROJECT_ROOT, RUN_ROOT,
    expected_dataset_sha256="f56625977fb6db8075ab620b1f3ef49a2a349ae75511097505855376e9684f91",
    expected_source_sha256="84eff3cde04c0e9a1756a3d29c44ec82e115a1a08575132686a02520333e2b5c",
    require_checkpoint=True,
)
os.chdir(PROJECT_ROOT)


In [ ]:
PIN_DEPENDENCIES = True
if PIN_DEPENDENCIES:
    subprocess.run([sys.executable, "-m", "pip", "install", "torch==2.8.0",
                    "--index-url", "https://download.pytorch.org/whl/cu126"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt",
                    "setuptools==78.1.0"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "--no-build-isolation", "-e", "."], check=True)
sys.path.insert(0, str(PROJECT_ROOT / "src"))
import torch
from sca.train import train, environment, write_json, code_identity
from sca.data import inspect_dataset
from sca.evaluate import evaluate_cached

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Select a GPU in Notebook Settings before the GPU experiment")
print(environment(torch.device("cuda")))
print(inspect_dataset(DATASET_PATH, verify_attack=False))
subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)
freeze = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
(Path("/kaggle/working") / "environment-freeze.txt").write_text(freeze)


In [ ]:
from pathlib import Path
import json, shutil, sys
extension_path=Path('/kaggle/working/kaggle_budget_diagnostic.py')
extension_path.write_text('"""A declared optimizer extension for one frozen larger-budget same-model Kaggle diagnostic."""\nimport hashlib\nimport json\nimport math\nimport time\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nfrom unittest.mock import patch\n\nimport numpy as np\nimport torch\n\nfrom sca.data import sha256_file, load_validation_evaluation, normalize\nfrom sca.evaluate import evaluate\nfrom sca.models import build_model\nfrom sca.train import code_identity, train, write_json\n\nBaseAdam = torch.optim.Adam\n\n\ndef author_cycle_lr(completed_updates, total_updates, maximum=0.005):\n    """Port the author\'s three linear phases, without momentum cycling."""\n    if total_updates < 5 or not 0 <= completed_updates <= total_updates:\n        raise ValueError("Invalid schedule position")\n    middle = int(total_updates * 0.8 / 2)\n    if completed_updates <= middle:\n        return maximum * 0.1 * (1 + 9 * completed_updates / middle)\n    if completed_updates <= 2 * middle:\n        return maximum * 0.1 * (1 + 9 * (1 - (completed_updates - middle) / middle))\n    return maximum * 0.1 * (1 - 0.99 * (completed_updates - 2 * middle) / (total_updates - 2 * middle))\n\n\nclass AuthorCycleAdam(BaseAdam):\n    """Keep Adam semantics and checkpoint the exact schedule position."""\n    def __init__(self, params, lr, *, total_updates, **kwargs):\n        super().__init__(params, lr=author_cycle_lr(0, total_updates, lr), **kwargs)\n        self.maximum = lr\n        self.total_updates = total_updates\n        self.completed_updates = 0\n        self.used_rates = []\n        self.started = time.perf_counter()\n\n    def step(self, closure=None):\n        if self.completed_updates >= self.total_updates:\n            raise RuntimeError("Frozen optimizer update budget exhausted")\n        if time.perf_counter() - self.started > 900:\n            raise RuntimeError("Frozen training-loop cost cap exceeded")\n        gradients = [p.grad.reshape(-1) for group in self.param_groups for p in group[\'params\'] if p.grad is not None]\n        if gradients and not bool(torch.isfinite(torch.cat(gradients)).all()):\n            raise FloatingPointError("Nonfinite gradients; no automatic retry")\n        self.used_rates.append(float(self.param_groups[0]["lr"]))\n        result = super().step(closure)\n        self.completed_updates += 1\n        for group in self.param_groups:\n            group["lr"] = author_cycle_lr(self.completed_updates, self.total_updates, self.maximum)\n        return result\n\n    def state_dict(self):\n        state = super().state_dict()\n        state["author_cycle"] = {"maximum": self.maximum, "total_updates": self.total_updates,\n            "completed_updates": self.completed_updates, "used_rates": self.used_rates.copy()}\n        return state\n\n    def load_state_dict(self, state):\n        state = dict(state)\n        cycle = state.pop("author_cycle")\n        if cycle["total_updates"] != self.total_updates or cycle["maximum"] != self.maximum:\n            raise ValueError("Schedule identity mismatch")\n        super().load_state_dict(state)\n        self.completed_updates = cycle["completed_updates"]\n        self.used_rates = cycle["used_rates"].copy()\n\n\ndef train_with_cycle(config, **kwargs):\n    total = config["epochs"] * math.ceil(config["n_train"] / config["batch_size"])\n    def optimizer_factory(params, lr):\n        return AuthorCycleAdam(params, lr, total_updates=total)\n    # The frozen production trainer remains unchanged; the extension is explicit in config/plan.\n    with patch("torch.optim.Adam", optimizer_factory):\n        return train(config, **kwargs)\n\n\ndef run_experiment(dataset, old_run, output, plan):\n    old_run, output = Path(old_run), Path(output)\n    if output.exists() and any(output.iterdir()):\n        raise FileExistsError("No automatic repeat or overwrite is authorized")\n    output.mkdir(parents=True)\n    started = time.perf_counter()\n    source = code_identity()["source_sha256"]\n    if source != plan["production_source_sha256"]:\n        raise ValueError("Production source changed")\n    if sha256_file(dataset) != plan["dataset_sha256"]:\n        raise ValueError("Dataset changed")\n    extension_hash = hashlib.sha256(Path(__file__).read_bytes()).hexdigest()\n    if extension_hash != plan["extension_sha256"]:\n        raise ValueError("Optimizer extension changed")\n    old_hashes = {name: sha256_file(old_run / name) for name in plan["baseline_hashes"]}\n    if old_hashes != plan["baseline_hashes"]:\n        raise ValueError("Baseline artifacts changed")\n    original = json.loads((old_run / "config.json").read_text(encoding="utf-8"))\n    config = dict(original)\n    config.update(dataset=str(dataset), run_dir=str(output / "none_seed0"),\n        n_train=45000, batch_size=50, learning_rate=0.005, device="cuda",\n        recipe_extension_sha256=extension_hash, recipe="author_linear_cycle_no_momentum")\n    expected = {"n_train":45000,"n_validation":5000,"epochs":50,"seed":0,\n        "split_seed":2026,"model":"cnn_literature","normalization":"feature_minmax_training_only"}\n    if any(config[k] != v for k,v in expected.items()) or config["augmentation"]["noise_std"] != 0 or config["augmentation"]["max_shift"] != 0:\n        raise ValueError("Frozen training scope mismatch")\n    write_json(output / "plan.json", plan)\n    write_json(output / "access.json", {"started_utc": datetime.now(timezone.utc).isoformat(),\n        "attack_payloads_read":False,"new_confirmation_rows":0,"new_training_started":True})\n    summary = train_with_cycle(config)\n    run = output / "none_seed0"\n    manifest = json.loads((run / "manifest.json").read_text())\n    old_manifest = json.loads((old_run / "manifest.json").read_text())\n    assert manifest["normalizer"]["kind"] == "feature_minmax_training_only"\n    assert manifest["split_id"] != old_manifest["split_id"]\n    assert manifest["parameters"] == 16952 and summary["optimization_steps"] == 45000\n    fresh = np.load(run / "splits.npz")\n    previous = np.load(old_run / "splits.npz")\n    assert np.array_equal(fresh["validation"], previous["validation"])\n    assert np.isin(previous["training"], fresh["training"]).all()\n    assert len(fresh["training"]) == 45000 and not np.intersect1d(fresh["training"], fresh["validation"]).size\n    evaluation_config = plan["evaluation_config"]\n    validation = evaluate(run, evaluation_config, device="cuda", split="validation", output_dir=output / "validation")\n    best = torch.load(run / "best.pt", map_location="cpu", weights_only=False)\n    traces, plaintext, key = load_validation_evaluation(dataset, np.load(run / "splits.npz")["validation"], 2)\n    traces = normalize(traces, best["normalizer"])\n    model = build_model("cnn_literature").cuda()\n    model.load_state_dict(best["model"])\n    model.eval()\n    with torch.no_grad():\n        lp = np.concatenate([torch.log_softmax(model(torch.from_numpy(traces[i:i+256]).cuda()).double(),dim=1).cpu().numpy()\n            for i in range(0,len(traces),256)])\n    np.save(output / "validation/log_probabilities.npy", lp)\n    np.save(output / "validation/plaintext_byte.npy", plaintext)\n    write_json(output / "validation/evaluation_key.json", {"true_key_byte":int(key),"evaluation_only":True})\n    state = torch.load(run / "last.pt", map_location="cpu", weights_only=False)\n    rates = state["optimizer"]["author_cycle"]["used_rates"]\n    np.save(output / "learning_rates.npy", rates)\n    assert len(rates) == 45000 and state["optimizer"]["author_cycle"]["completed_updates"] == 45000\n    criterion = validation[0]["sr_at_budget"] >= 0.9\n    after = {name: sha256_file(old_run / name) for name in old_hashes}\n    assert after == old_hashes and code_identity()["source_sha256"] == source\n    result = {"complete":True,"primary_met":bool(criterion),"validation":validation,"summary":summary,\n        "experiment_seconds":time.perf_counter()-started,"new_gpu_trainings":1,"full_gpu_trainings_total":5,\n        "combined_training_performed":False,"original_cnn_gate_unchanged":True,"attack_payloads_read":False,\n        "new_confirmation_rows":0,"new_seeds":0,"same_epochs_and_trace_exposures":False, "same_epochs":True,\n        "update_budget_changed":True,"baseline_updates":10000,"new_updates":45000, "training_exposures":2250000,\n        "production_source_preserved":True,"baseline_hashes":after,"extension_sha256":extension_hash,\n        "selection_rule":"minimum validation CE","interpretation":"Joint unique-sample/update-count/refitted-scaler diagnostic; viewed validation, no independent efficacy or isolated-cause claim"}\n    write_json(output / "results.json", result)\n    print(json.dumps(result, indent=2), flush=True)\n    return result\n',encoding='utf-8')
sys.path.insert(0,'/kaggle/working')
from kaggle_budget_diagnostic import run_experiment
frozen_plan=json.loads('{"authorization": "User explicitly approved one baseline45k and cap5 on2026-10-11; no combined or attack authorization", "hypothesis": "Same adapted CNN/author cycle with45k training restores clean validation SR>=18/20 at2000", "primary_endpoint": "Minimum-CE checkpoint clean validation SR@2000 over same20 trace orders", "primary_criterion": "SR>=0.90; all20 orders retained, no alternative success threshold", "secondary": "CE/accuracy/history/GE/sustainedSR and saved predictions, descriptive only", "baseline": "Completed recipe diagnostic10k/batch50/cycle/seed0/50epochs; reference old source checkpoint remains preserved", "alternative": "Same16952parameterCNN/seed0/50epochs/batch50/cycle, nested45k training and refitted training-only MinMax", "schedule_knot_updates": [0, 18000, 36000, 45000], "schedule_knot_lr": [0.0005, 0.005, 0.0005, 5e-06], "budget": "45000training/5000sharedvalidation/50epochs/2250000exposures/45000updates versus500000exposures/10000updates", "confounds": "Joint unique-sample/update-count/refitted-scaler diagnostic; not equal-compute or exact Keras reproduction", "validation_status": "Previously viewed shared original5k profiling validation, not fresh confirmation. Its CE selects epoch; no efficacy claim on independent attack", "training_information": "Original identity labels; train-only per-position MinMax; model input traces alone", "new_confirmation_rows": 0, "variable_unused_rows_unchanged": 30000, "attack_payloads_read": false, "new_seeds": 0, "new_models": 0, "new_gpu_trainings_max": 1, "full_gpu_trainings_total_after_success": 5, "combined_training_authorized": false, "original_cnn_gate_preserved": true, "notebook_ref": "nmavros/hw-sec-exp2", "stopping_rule": "One submission and one50epoch training; no HPO, restart, schedule search, extra seed or automatic combined. Stop on correctness/hash/nonfinite failure; preserve partial checkpoint and all negatives", "session_timeout_seconds": 1800, "training_loop_expected_cap_seconds": 900, "production_source_sha256": "84eff3cde04c0e9a1756a3d29c44ec82e115a1a08575132686a02520333e2b5c", "dataset_sha256": "f56625977fb6db8075ab620b1f3ef49a2a349ae75511097505855376e9684f91", "extension_sha256": "35faf762350da2e3d1b31b38a84907052b7abe95c91f9c3e49e1c0a7a8620768", "baseline_hashes": {"best.pt": "c374ff2dcf7e523341069f7f2e2a9c001f7e81d3ff94039af2de6dda1863ba64", "last.pt": "48d09b8aae9487fecafb33f4b5bd940cf1ab7f3cbdee9a519c61c44508165284", "manifest.json": "1b159e5af59e4beb03a96762010498da88a2974e378dc171e61d785432e20f0c", "splits.npz": "b45dcb830d2687163a77db17d5e4caff40545344c6f7dba9b5bc1bf414c88347", "history.json": "7ea9e0d51dd96f6337ab54138ece8a4279a0e0d9e38ab00e5055e1563c5861e3", "config.json": "49518da6cf789fd2ecd02e74d5a4146aa17626c3ee17f77d6000720d37d71677"}, "input_hashes": {"notebooks/kaggle_baseline.ipynb": "538c8d30ad2dc01611629522347068806374ef2a01674d00ce408522a9b6bd0e", "outputs/kaggle_resume_input_literature.zip": "f39f82a77cacbe64d024d5f40d4f858cae90f95452e355ebb49b015cae513c56", "notebooks/verify_budget_diagnostic.py": "2c3dc81651eda3b90ac2846975ad547900540562c8fa967fc7f051566ad0bedd", "tests/test_kaggle_budget_diagnostic.py": "37f612fb6e8546cf567c49bf4d372843af4b003fa8d57c320eb49cca10556dd7", "outputs/kaggle_budget_2026-10-11/pytest.xml": "8289370b1ca3bf8e6a360ac2adccce95311a2f9352b1584e54b1900b338c601e", "docs/PAPER_NEXT_PROTOCOL_2026-10-11.md": "e35fc6737d8660f701dfd59794970e2d630ea71fad8d67b03ba243ad233eacdb", "outputs/kaggle_recipe_2026-10-09/results.json": "e22cc1ed501fc1a28181bae1a7522d32b30a40c43c7a4271cae01ff0e509cc2d", "outputs/paper_readiness_2026-10-11/audit.json": "3d942960415cebd076920d836115a3167c8fee89c89825792c0408067a18592c"}, "evaluation_config": {"budget": 2000, "repetitions": 20, "success_threshold": 0.9, "attack_order_seed": 8001, "corruption_seed": 9001, "batch_size": 256, "conditions": [{"name": "clean", "noise_std": 0.0, "max_shift": 0, "padding": "zero"}]}, "tests": {"name": "pytest", "errors": "0", "failures": "0", "skipped": "0", "tests": "104", "time": "17.136", "timestamp": "2026-10-11T13:16:45.741705+03:00", "hostname": "mavroul1s"}, "frozen_utc": "2026-10-11T10:17:34.678397+00:00", "novelty_verified": false, "notebook_ref_status": "Intended manual-upload title; actual URL must be recorded after creation", "original_frozen_plan_sha256": "a39c2e4cf476239062ce3ea420f056b8b844cea2b91407c228bb1ce470e1b10f", "account_revision": {"previous_owner": "con1los", "authorized_owner": "nmavros", "mode": "Manual upload; one active continuation notebook in the new account", "authorization": "User requested using nmavros and asked to prepare the notebook for manual execution", "scientific_protocol_changed": false, "new_training_authorization": false, "original_private_notebook_preserved": true}, "input_delivery": "Upload unchanged kaggle_resume_input_literature.zip as a private Dataset in nmavros and attach it", "manual_export_utc": "2026-10-11T10:28:57.535418+00:00"}')
from sca.train import write_json
experiment_output=Path('/kaggle/working/budget_diagnostic')
try:
    result=run_experiment(DATASET_PATH,RUN_ROOT/'none_seed0',Path('/kaggle/working/budget_diagnostic'),frozen_plan)
except Exception as error:
    experiment_output.mkdir(parents=True,exist_ok=True)
    write_json(experiment_output/'execution_failure.json',{'complete':False,'error_type':type(error).__name__,'message':str(error),'automatic_retry':False})
    raise
finally:
    if experiment_output.exists():
        write_json(experiment_output/'account_revision.json',frozen_plan['account_revision'])
        shutil.make_archive('/kaggle/working/budget_diagnostic_results','zip',experiment_output)
        print('Download from Output: budget_diagnostic_results.zip')
